# Scenario 5: SCD 2

A retail company maintains a product catalog in its data warehouse. Product details such as name, category, and price may change over time due to rebranding, category updates, or pricing adjustments. To preserve historical data for accurate reporting and trend analysis, the company needs to implement a Slowly Changing Dimension (SCD) Type 2 mechanism in PySpark, ensuring old records are retained with effective date ranges while new versions are inserted as separate records.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import *

In [0]:
%sql
SELECT * 
FROM pyspark_catalog.source.customers        -- catalog -> schema -> table

id,email,city,country,modifiedDate
1,john.smith@example.com,New York,USA,2026-08-23T16:03:59.201Z
2,jane.doe@example.com,London,UK,2026-08-23T16:03:59.201Z
3,mike.williams@example.com,Paris,France,2026-08-23T16:03:59.201Z
4,sara.jones@example.com,Tokyo,Japan,2026-08-23T16:03:59.201Z
5,peter.chen@example.com,Sydney,Australia,2026-08-23T16:03:59.201Z
1,john.smith@example.com,Ho Chi Minh City,Vietnam,2026-08-23T16:46:47.408Z
6,hoho.doe@example.com,London,UK,2026-08-23T16:46:47.408Z


In [0]:
if not spark.catalog.tableExists("pyspark_catalog.source.DimCustomers"):
    spark.sql("""
            CREATE TABLE IF NOT EXISTS pyspark_catalog.source.DimCustomers
            SELECT 
              *,
              current_timestamp() AS start_date,
              CAST('3000-01-01' AS timestamp) AS end_date,
              'Y' as is_active
            FROM pyspark_catalog.source.customers
              """)

else:
    pass

In [0]:
%sql
-- This is the TARGET
SELECT * FROM pyspark_catalog.source.DimCustomers

id,email,city,country,modifiedDate,start_date,end_date,is_active
1,john.smith@example.com,New York,USA,2026-08-23T16:03:59.201Z,2026-08-23T16:05:34.765Z,3000-01-01T00:00:00.000Z,Y
2,jane.doe@example.com,London,UK,2026-08-23T16:03:59.201Z,2026-08-23T16:05:34.765Z,3000-01-01T00:00:00.000Z,Y
3,mike.williams@example.com,Paris,France,2026-08-23T16:03:59.201Z,2026-08-23T16:05:34.765Z,3000-01-01T00:00:00.000Z,Y
4,sara.jones@example.com,Tokyo,Japan,2026-08-23T16:03:59.201Z,2026-08-23T16:05:34.765Z,3000-01-01T00:00:00.000Z,Y
5,peter.chen@example.com,Sydney,Australia,2026-08-23T16:03:59.201Z,2026-08-23T16:05:34.765Z,3000-01-01T00:00:00.000Z,Y


## SCD_TYPE 2

### Create and Dedup the SOURCE

In [0]:
# Create the source
source = spark.sql("SELECT * FROM pyspark_catalog.source.customers")

# Dedup the source
source = source.withColumn("dedup", row_number().over(Window
                                                      .partitionBy("id")
                                                      .orderBy(desc("modifiedDate"))))
source = source.filter(col('dedup') == 1).drop("dedup")

# Create "src_temp" view
source.createOrReplaceTempView('src_temp')

# Create the "real_source" view
source = spark.sql("""
                   SELECT 
                        *,
                        current_timestamp() AS start_date,
                        CAST('3000-01-01' AS timestamp) AS end_date,
                        'Y' as is_active
                    FROM src_temp
                   """)
source.createOrReplaceTempView('real_source')

In [0]:
%sql
SELECT * FROM real_source

id,email,city,country,modifiedDate,start_date,end_date,is_active
1,john.smith@example.com,Ho Chi Minh City,Vietnam,2026-08-23T16:46:47.408Z,2026-08-23T16:52:41.792Z,3000-01-01T00:00:00.000Z,Y
2,jane.doe@example.com,London,UK,2026-08-23T16:03:59.201Z,2026-08-23T16:52:41.792Z,3000-01-01T00:00:00.000Z,Y
3,mike.williams@example.com,Paris,France,2026-08-23T16:03:59.201Z,2026-08-23T16:52:41.792Z,3000-01-01T00:00:00.000Z,Y
4,sara.jones@example.com,Tokyo,Japan,2026-08-23T16:03:59.201Z,2026-08-23T16:52:41.792Z,3000-01-01T00:00:00.000Z,Y
5,peter.chen@example.com,Sydney,Australia,2026-08-23T16:03:59.201Z,2026-08-23T16:52:41.792Z,3000-01-01T00:00:00.000Z,Y
6,hoho.doe@example.com,London,UK,2026-08-23T16:46:47.408Z,2026-08-23T16:52:41.792Z,3000-01-01T00:00:00.000Z,Y


### MERGE 1: Marking the updated records as expired

In [0]:
%sql
MERGE INTO pyspark_catalog.source.DimCustomers AS target
USING real_source
ON target.id = real_source.id
AND target.is_active = "Y"

WHEN MATCHED AND (                              -- Are these different, even if NULL?
    NOT (real_source.email <=> target.email) OR
    NOT (real_source.city <=> target.city) OR
    NOT (real_source.country <=> target.country)
) THEN UPDATE SET
    target.end_date = current_timestamp(),
    target.is_active = "N"

-- null-safe equality operator helps correctly handles:
-- NULL → value       = change
-- value → NULL       = change
-- NULL → NULL        = no change
-- value → same value = no change

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


### MERGE 2: When records not matched, then insert new + update records

In [0]:
%sql
MERGE INTO pyspark_catalog.source.DimCustomers AS target
USING real_source
ON target.id = real_source.id
AND target.is_active = "Y"

WHEN NOT MATCHED THEN INSERT 
(
    id, 
    email,
    city,
    country,
    modifiedDate,
    start_date,
    end_date,
    is_active
)
VALUES (
    real_source.id,
    real_source.email,
    real_source.city,
    real_source.country,
    real_source.modifiedDate,
    real_source.start_date,
    real_source.end_date,
    real_source.is_active
)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


In [0]:
%sql
-- Inspect the TARGET the last time to ensur success
SELECT * FROM pyspark_catalog.source.DimCustomers

id,email,city,country,modifiedDate,start_date,end_date,is_active
2,jane.doe@example.com,London,UK,2026-08-23T16:03:59.201Z,2026-08-23T16:05:34.765Z,3000-01-01T00:00:00.000Z,Y
3,mike.williams@example.com,Paris,France,2026-08-23T16:03:59.201Z,2026-08-23T16:05:34.765Z,3000-01-01T00:00:00.000Z,Y
4,sara.jones@example.com,Tokyo,Japan,2026-08-23T16:03:59.201Z,2026-08-23T16:05:34.765Z,3000-01-01T00:00:00.000Z,Y
5,peter.chen@example.com,Sydney,Australia,2026-08-23T16:03:59.201Z,2026-08-23T16:05:34.765Z,3000-01-01T00:00:00.000Z,Y
1,john.smith@example.com,New York,USA,2026-08-23T16:03:59.201Z,2026-08-23T16:05:34.765Z,2026-08-23T16:53:44.123Z,N
1,john.smith@example.com,Ho Chi Minh City,Vietnam,2026-08-23T16:46:47.408Z,2026-08-23T16:53:57.314Z,3000-01-01T00:00:00.000Z,Y
6,hoho.doe@example.com,London,UK,2026-08-23T16:46:47.408Z,2026-08-23T16:53:57.314Z,3000-01-01T00:00:00.000Z,Y


### Insert new records to test

In [0]:
%sql
-- Change the value and add new record
INSERT INTO pyspark_catalog.source.customers
VALUES
('1', 'john.smith@example.com', 'Ho Chi Minh City', 'Vietnam', current_timestamp()),
('6', 'hoho.doe@example.com', 'London', 'UK', current_timestamp())

num_affected_rows,num_inserted_rows
2,2
